In [ ]:
# This script calculates heat season characteristics including: onset date, cessation date, and length.
# In the manuscript, it is used for calculating heat season characteristics for the baseline period analysis of 1966-1995.

In [ ]:
import glob
import os
import re
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks

In [ ]:
# Location of the original daily temperature data from the CESM2-LE. 
# Ths script will sort through all ensemble members, but only handles one temperature variable (TMAX or TMIN) at a time.

#DATA_DIR = '/glade/campaign/collections/gdex/data/d651056/CESM2-LE/atm/proc/tseries/day_1/TREFHTMN/'
DATA_DIR = '/glade/campaign/collections/gdex/data/d651056/CESM2-LE/atm/proc/tseries/day_1/TREFHTMX/'


# Location where the netcdf files containing the calculated heat season characterisitcs will go
#OUTPUT_DIR = '/.../TMMN/...'
OUTPUT_DIR = '/.../TMAX/...'


# --- A. FILE SEARCH PARAMETERS ---
FILE_HIST_START, FILE_HIST_END = 1961, 2014
FILE_FUT_START, FILE_FUT_END = 2015, 2060

# --- B. ANALYSIS PARAMETERS (SLICING) ---
# These are the exact dates used for the science
# Example: 1961-1990 Baseline vs 1991-2020 Future
ANALYSIS_HIST_START, ANALYSIS_HIST_END = 1966, 1995
#ANALYSIS_FUT_START,  ANALYSIS_FUT_END  = 1996, 2025
ANALYSIS_FUT_START,  ANALYSIS_FUT_END  = 2026, 2055


#variable_name = 'TREFHTMN'
variable_name = 'TREFHTMX'

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def find_ensemble_members(data_dir):
    """
    Scans directory for unique ensemble member IDs (e.g., '1011.001').
    """
    #all_files = glob.glob(os.path.join(data_dir, "*TREFHTMN*.nc")) # need to select whichever file is consistent with the directory above
    all_files = glob.glob(os.path.join(data_dir, "*TREFHTMX*.nc"))
    member_ids = set()
    pattern = re.compile(r'smbb.f09_g17.LE2-(\d{4}\.\d{3})')
    
    for f in all_files:
        match = pattern.search(os.path.basename(f))
        if match:
            member_ids.add(match.group(1))
    return sorted(list(member_ids))

def get_cesm_files(data_dir, member_id, start_year, end_year, scenario_tag):
    """
    Finds files for a specific member that OVERLAP with the requested period.
    """
    glob_pattern = os.path.join(data_dir, f"*{scenario_tag}*LE2-{member_id}*.nc")
    candidate_files = sorted(glob.glob(glob_pattern))
    selected_files = []
    
    for f in candidate_files:
        try:
            date_part = f.replace('.nc', '').split('.')[-1]
            if '-' in date_part:
                f_start_str, f_end_str = date_part.split('-')
                f_start_yr = int(f_start_str[:4])
                f_end_yr = int(f_end_str[:4])
                
                if not (f_end_yr < start_year or f_start_yr > end_year):
                    selected_files.append(f)
        except (ValueError, IndexError):
            continue
            
    return selected_files

In [ ]:
def calculate_threshold(data, start_year, end_year, percentile=0.97):
    """
    Calculates the threshold using ONLY the analysis years.
    """
    print(f"Calculating {int(percentile*100)}th percentile threshold for {start_year}-{end_year}...")
    
    analysis_slice = data.sel(time=slice(str(start_year), str(end_year)))
    threshold = analysis_slice.quantile(percentile, dim='time').compute()
    return threshold


def map_tropical_bimodality(raw_data, threshold_map, 
                            min_peak_distance=90,        
                            prominence_fraction=0.10, 
                            max_valley_ratio=0.30,
                            min_years_per_peak=10,        
                            peak_window_days=30):        
    """
    Creates maps of extreme heat seasonality, including the number of modes,
    the shallow valley DOY (for S1/S2 splits), and the deep valley month
    (for explicitly locking the calendar boundary).
    """
    print(f"Starting Bimodality & Valley Mapping...")
    print(f"  Parameters: Distance > {min_peak_distance}d, Prominence > {prominence_fraction*100}%, Valley Depth < {max_valley_ratio*100}%")
    
    lat_bound = 23.5
    if raw_data.lat[0] > raw_data.lat[-1]:
        trop_slice = slice(lat_bound, -lat_bound)
    else:
        trop_slice = slice(-lat_bound, lat_bound)
        
    data_trop = raw_data.sel(lat=trop_slice)
    thresh_trop = threshold_map.sel(lat=trop_slice)
    
    print("  Calculating extreme days mask (this may take a moment)...")
    is_extreme = (data_trop > thresh_trop).compute().values
    
    lats = data_trop.lat.values
    lons = data_trop.lon.values
    doys = data_trop.time.dt.dayofyear.values
    years = data_trop.time.dt.year.values 
    
    peak_count_map = np.full((len(lats), len(lons)), np.nan)
    valley_doy_map = np.full((len(lats), len(lons)), np.nan) 
    deep_valley_month_map = np.full((len(lats), len(lons)), np.nan)
    
    x_eval = np.arange(1, 366)
    
    for i, lat in enumerate(lats):
        if i % 10 == 0: 
            print(f"    Processing Latitude {i}/{len(lats)} ({lat:.2f}°)...", end='\r')
            
        for j, lon in enumerate(lons):
            pixel_extremes = is_extreme[:, i, j]
            if not np.any(pixel_extremes):
                continue
                
            pixel_doys = doys[pixel_extremes]
            pixel_years = years[pixel_extremes] 
            
            if len(pixel_doys) < 30:
                peak_count_map[i, j] = 1 
                continue
                
            doys_extended = np.concatenate([pixel_doys - 365, pixel_doys, pixel_doys + 365])
            
            try:
                kde = gaussian_kde(doys_extended, bw_method=0.05)
                density = kde.evaluate(x_eval)
                
                max_density = np.max(density)
                min_prominence = max_density * prominence_fraction
                peaks, _ = find_peaks(density, distance=min_peak_distance, prominence=min_prominence)
                
                # DISTANCE & VALLEY CHECK
                valid_peaks = list(peaks)
                merged = True
                
                while merged and len(valid_peaks) > 1:
                    merged = False
                    valid_peaks.sort() 
                    
                    for k in range(len(valid_peaks)):
                        p1 = valid_peaks[k]
                        p2 = valid_peaks[(k + 1) % len(valid_peaks)]
                        
                        dist = min((p2 - p1) % 365, (p1 - p2) % 365)
                        if dist < min_peak_distance:
                            drop_p = p1 if density[p1] < density[p2] else p2
                            valid_peaks.remove(drop_p)
                            merged = True
                            break 
                        
                        if p1 < p2:
                            valley = np.min(density[p1:p2])
                        else:
                            valley = min(np.min(density[p1:]), np.min(density[:p2]))
                            
                        smaller_peak = min(density[p1], density[p2])
                        
                        if valley > smaller_peak * max_valley_ratio:
                            drop_p = p1 if density[p1] < density[p2] else p2
                            valid_peaks.remove(drop_p)
                            merged = True
                            break 

                # TEMPORAL CONSISTENCY CHECK
                robust_peaks = []
                for p in valid_peaks:
                    p_doy = x_eval[p]
                    dist = np.minimum((pixel_doys - p_doy) % 365, (p_doy - pixel_doys) % 365)
                    contributing_years = np.unique(pixel_years[dist <= peak_window_days])
                    if len(contributing_years) >= min_years_per_peak:
                        robust_peaks.append(p)
                        
                valid_peaks = robust_peaks
                num_peaks = max(1, len(valid_peaks))
                peak_count_map[i, j] = num_peaks

                # IDENTIFY SHALLOW AND DEEP VALLEYS
                if num_peaks >= 2:
                    valid_peaks.sort()
                    p1 = valid_peaks[0]
                    p2 = valid_peaks[1] 
                    
                    v1_idx = np.argmin(density[p1:p2]) + p1
                    v1_density = density[v1_idx]
                    
                    idx_after = np.argmin(density[p2:]) + p2
                    idx_before = np.argmin(density[:p1])
                    
                    if density[idx_after] < density[idx_before]:
                        v2_idx = idx_after
                    else:
                        v2_idx = idx_before
                    v2_density = density[v2_idx]
                    
                    # Explicitly define Shallow vs. Deep
                    if v1_density > v2_density:
                        shallow_idx = v1_idx
                        deep_idx = v2_idx
                    else:
                        shallow_idx = v2_idx
                        deep_idx = v1_idx
                        
                    # Assign the Shallow Valley DOY for the S1/S2 Split
                    valley_doy_map[i, j] = x_eval[shallow_idx]
                    
                    # Assign the Deep Valley Month for the Calendar Split
                    deep_doy = int(x_eval[deep_idx])
                    deep_month = (datetime.datetime(2001, 1, 1) + datetime.timedelta(days=deep_doy - 1)).month
                    deep_valley_month_map[i, j] = deep_month
                
            except Exception as e:
                continue

    print(f"\n  Done. Map generation complete.        ")
    
    ds_out = xr.Dataset(
        {
            'season_modes': (['lat', 'lon'], peak_count_map),
            'valley_doy': (['lat', 'lon'], valley_doy_map),
            'deep_valley_month': (['lat', 'lon'], deep_valley_month_map)
        },
        coords={'lat': lats, 'lon': lons}
    )
    
    return ds_out



# =============================================================================
# 3. METRICS CALCULATION 
# =============================================================================
def get_year_metrics(data_slice, threshold, year, is_southern):
    """
    Calculates start, end, and duration.
    Wraps start/end dates back to 1-365 range for plotting.
    Calendar-aware to handle CESM 'noleap' data safely.
    """
    doy = data_slice.time.dt.dayofyear
    
    # Calendar-Aware Year Length
    calendar = getattr(data_slice.time.dt, 'calendar', 'standard')
    if calendar in ['noleap', '365_day']:
        days_in_year = 365
    else:
        days_in_year = 366 if pd.Timestamp(f"{year}-12-31").is_leap_year else 365

    if is_southern:
        adjusted_doy = xr.where(doy < 183, doy + days_in_year, doy)
    else:
        adjusted_doy = doy

    is_extreme = data_slice >= threshold
    extreme_days = adjusted_doy.where(is_extreme)

    f = extreme_days.min(dim='time')
    l = extreme_days.max(dim='time')
    
    length = (l - f) + 1
    length = length.fillna(0) 

    f_clean = (f - 1) % days_in_year + 1
    l_clean = (l - 1) % days_in_year + 1

    return f_clean, l_clean, length


def calculate_period_stats(data, threshold, start_year, end_year):
    """
    Loops through analysis years and computes metrics for NH and SH.
    """
    years = np.arange(start_year, end_year + 1)
    
    nh_first, nh_last, nh_len = [], [], []
    sh_first, sh_last, sh_len = [], [], []

    nh_thresh = threshold.where(threshold.lat >= 0, drop=True)
    sh_thresh = threshold.where(threshold.lat < 0, drop=True)

    print(f"  Computing metrics for analysis period {start_year}-{end_year}...")
    
    for year in years:
        try:
            nh_data = data.sel(time=str(year)).where(data.lat >= 0, drop=True)
            if nh_data.time.size > 0:
                f, l, d = get_year_metrics(nh_data, nh_thresh, year, is_southern=False)
                nh_first.append(f.compute())
                nh_last.append(l.compute())
                nh_len.append(d.compute())
        except KeyError: pass

        try:
            prev_year = year - 1
            t_start, t_end = f"{prev_year}-07-01", f"{year}-06-30"
            
            sh_data = data.sel(time=slice(t_start, t_end)).where(data.lat < 0, drop=True)
            
            if sh_data.time.size > 300: 
                f, l, d = get_year_metrics(sh_data, sh_thresh, year, is_southern=True)
                sh_first.append(f.compute())
                sh_last.append(l.compute())
                sh_len.append(d.compute())
        except KeyError: pass
        
        print(f"    Processed {year}", end='\r')
    print("")

    return (nh_first, nh_last, nh_len, sh_first, sh_last, sh_len)


# =============================================================================
# TROPICAL SLICING
# =============================================================================
def get_tropical_subset(data, lat_bound=23.5):
    lat_values = data.lat.values
    if lat_values[0] > lat_values[-1]:
        return data.sel(lat=slice(lat_bound, -lat_bound))
    else:
        return data.sel(lat=slice(-lat_bound, lat_bound))


# =============================================================================
# DISCOVERY: START MONTH (Deepest Minimum Window)
# =============================================================================
def get_tropical_start_month(trop_data, trop_threshold):
    """
    Determines the optimal start month for tropical pixels using the
    5-month rolling sum technique.
    """
    print("  Determining start months (Deepest Minimum Window)...")
    
    is_extreme = trop_data > trop_threshold
    monthly_clim = is_extreme.groupby('time.month').sum(dim='time')
    
    clim_padded = xr.concat([monthly_clim, monthly_clim, monthly_clim], dim='month')
    rolling_sum = clim_padded.rolling(month=5, center=True).sum()
    middle_rolling = rolling_sum.isel(month=slice(12, 24))
    
    best_idx = middle_rolling.argmin(dim='month')
    start_month = best_idx + 1
    
    return start_month.fillna(1).astype(int).drop_vars('time', errors='ignore')


def calculate_tropical_metrics(data, threshold, start_year, end_year, 
                               ref_start_month_map=None,
                               bimodal_modes_map=None,  
                               bimodal_valley_map=None,
                               bimodal_start_month_map=None): 
    """
    Calculates flexible metrics ONLY for the tropics.
    Overrides the rolling-sum map with the KDE deep valley map for bimodal pixels.
    """
    print(f"Starting Tropical Analysis ({start_year}-{end_year})...")
    
    data_trop = get_tropical_subset(data)
    thresh_trop = get_tropical_subset(threshold)
    
    # --- MERGE START MONTH MAPS ---
    if ref_start_month_map is not None:
        start_month_map = get_tropical_subset(ref_start_month_map)
    else:
        base_start_month = get_tropical_start_month(data_trop, thresh_trop)
        
        if bimodal_modes_map is not None and bimodal_start_month_map is not None:
            bimodal_override = get_tropical_subset(bimodal_start_month_map)
            modes_map_trop = get_tropical_subset(bimodal_modes_map)
            
            start_month_map = xr.where(modes_map_trop >= 2, bimodal_override, base_start_month)
            start_month_map = start_month_map.fillna(1).astype(int)
        else:
            start_month_map = base_start_month
            
    # --- BIMODAL MAPS ASSIGNMENT ---
    if bimodal_modes_map is not None and bimodal_valley_map is not None:
        modes_map = get_tropical_subset(bimodal_modes_map)
        valley_map = get_tropical_subset(bimodal_valley_map)
    else:
        modes_map = xr.zeros_like(start_month_map) + 1
        valley_map = xr.full_like(start_month_map, np.nan)
    
    results_list = []
    
    month_days = {1:31, 2:28, 3:31, 4:30, 5:31, 6:30, 7:31, 8:31, 9:30, 10:31, 11:30, 12:31}
    calendar = getattr(data_trop.time.dt, 'calendar', 'standard')

    for m in range(1, 13):
        mask = (start_month_map == m).compute()
        if not mask.any(): continue
            
        print(f"  Computing Start Month {m}...", end='\r')
        
        group_data = data_trop.where(mask, drop=True)
        group_thresh = thresh_trop.where(mask, drop=True)
        group_modes = modes_map.where(mask, drop=True)
        group_valley = valley_map.where(mask, drop=True)
        
        years = np.arange(start_year, end_year + 1)
        
        for yr in years:
            if calendar in ['noleap', '365_day']:
                days_in_year = 365
                is_leap = False
            else:
                is_leap = pd.Timestamp(f"{yr}-12-31").is_leap_year
                days_in_year = 366 if is_leap else 365

            if m == 1:
                t_start, t_end = f"{yr}-01-01", f"{yr}-12-31"
            else:
                prev_m = m - 1
                end_day = month_days[prev_m]
                if is_leap and prev_m == 2:
                    end_day = 29
                    
                t_start = f"{yr-1}-{m:02d}-01"
                t_end = f"{yr}-{prev_m:02d}-{end_day}"

            try:
                yr_slice = group_data.sel(time=slice(t_start, t_end))
            except KeyError: continue

            if yr_slice.time.size < 300: continue

            raw_doy = yr_slice.time.dt.dayofyear
            is_extreme = yr_slice > group_thresh
            extreme_days = raw_doy.where(is_extreme)
            
            if m != 1:
                approx_start_doy = (m - 1) * 30 
                shifted_extremes = xr.where(extreme_days < approx_start_doy, 
                                            extreme_days + days_in_year, 
                                            extreme_days)
                shifted_valley = xr.where(group_valley < approx_start_doy,
                                          group_valley + days_in_year,
                                          group_valley)
            else:
                shifted_extremes = extreme_days
                shifted_valley = group_valley
            
            # --- Unimodal Boundaries ---
            f_uni = shifted_extremes.min(dim='time')
            l_uni = shifted_extremes.max(dim='time')
            len_uni = (l_uni - f_uni) + 1
            
            # --- Bimodal Boundaries ---
            s1_days = shifted_extremes.where(shifted_extremes <= shifted_valley)
            s2_days = shifted_extremes.where(shifted_extremes > shifted_valley)
            
            s1_f = s1_days.min(dim='time')
            s1_l = s1_days.max(dim='time')
            s2_f = s2_days.min(dim='time')
            s2_l = s2_days.max(dim='time')
            
            len1 = (s1_l - s1_f) + 1
            len2 = (s2_l - s2_f) + 1
            len_bimodal = len1.fillna(0) + len2.fillna(0)
            
            is_bimodal_pixel = group_modes >= 2
            
            final_len = xr.where(is_bimodal_pixel, len_bimodal, len_uni)
            final_len = final_len.fillna(0).drop_vars('time', errors='ignore').compute()
            
            def clean_doy(da):
                return (da.compute() - 1) % days_in_year + 1
                
            s1_start = xr.where(is_bimodal_pixel, clean_doy(s1_f), clean_doy(f_uni))
            s1_end   = xr.where(is_bimodal_pixel, clean_doy(s1_l), clean_doy(l_uni))
            
            s2_start = xr.where(is_bimodal_pixel, clean_doy(s2_f), np.nan)
            s2_end   = xr.where(is_bimodal_pixel, clean_doy(s2_l), np.nan)
            
            def expand(da):
                return da.reindex_like(start_month_map).expand_dims(year=[yr])
                
            results_list.append({
                'year': yr, 'len': expand(final_len), 
                's1_start': expand(s1_start), 's1_end': expand(s1_end),
                's2_start': expand(s2_start), 's2_end': expand(s2_end)
            })

    print("")
    print("  Aggregating final tropical results...")
    
    def combine_list(key):
        return xr.concat([item[key] for item in results_list], dim='temp').groupby('year').mean(dim='temp')

    final_len = xr.concat([item['len'] for item in results_list], dim='temp').groupby('year').sum(dim='temp')
    final_s1_start = combine_list('s1_start')
    final_s1_end = combine_list('s1_end')
    final_s2_start = combine_list('s2_start')
    final_s2_end = combine_list('s2_end')
    
    return final_len, final_s1_start, final_s1_end, final_s2_start, final_s2_end, start_month_map

In [ ]:
def run_unified_global_analysis(data, threshold, start_year, end_year, output_file="CESM_Global_Extreme_Heat_TMAX_Unified.nc", 
                                ref_start_month_map=None, bimodal_modes_map=None, bimodal_valley_map=None,
                                bimodal_start_month_map=None, extra_threshold=None):
    
    print(f"=== STARTING UNIFIED GLOBAL ANALYSIS ({start_year}-{end_year}) ===")

    print("\n>>> Phase 1: Tropical Calculation (+/- 23.5 Lat)")
    trop_len, trop_s1_start, trop_s1_end, trop_s2_start, trop_s2_end, trop_map = calculate_tropical_metrics(
        data, threshold, start_year, end_year, 
        ref_start_month_map=ref_start_month_map,
        bimodal_modes_map=bimodal_modes_map,  
        bimodal_valley_map=bimodal_valley_map,
        bimodal_start_month_map=bimodal_start_month_map
    )

    print("\n>>> Phase 2: Extra-Tropical Calculation (NH/SH)")
    et_stats = calculate_period_stats(data, threshold, start_year, end_year)
    
    nh_start_list, nh_end_list, nh_len_list = et_stats[0], et_stats[1], et_stats[2]
    sh_start_list, sh_end_list, sh_len_list = et_stats[3], et_stats[4], et_stats[5]

    def list_to_3d(nh_list, sh_list):
        nh_da = xr.concat(nh_list, dim='year')
        sh_da = xr.concat(sh_list, dim='year')
        return xr.concat([nh_da, sh_da], dim='lat').sortby('lat')

    et_len = list_to_3d(nh_len_list, sh_len_list)
    et_s1_start = list_to_3d(nh_start_list, sh_start_list)
    et_s1_end = list_to_3d(nh_end_list, sh_end_list)
    
    et_s2_start = xr.full_like(et_s1_start, np.nan)
    et_s2_end = xr.full_like(et_s1_end, np.nan)

    print("\n>>> Phase 3: Merging & Saving")
    
    clean_threshold = threshold.drop_vars(['quantile', 'time'], errors='ignore')

    output_vars = {
        'season_length': et_len,
        's1_start': et_s1_start,
        's1_end': et_s1_end,
        's2_start': et_s2_start,
        's2_end': et_s2_end,
        'threshold': clean_threshold
    }
    
    if extra_threshold is not None:
        clean_extra = extra_threshold.drop_vars(['quantile', 'time'], errors='ignore')
        output_vars['future_threshold'] = clean_extra

    if trop_map is not None:
        clean_trop_map = trop_map.drop_vars(['time'], errors='ignore')
        output_vars['start_month'] = clean_trop_map
    
    ds_out = xr.Dataset(output_vars)
    
    lat_bound = 23.5
    trop_slice = slice(-lat_bound, lat_bound) if ds_out.lat[0] < ds_out.lat[-1] else slice(lat_bound, -lat_bound)

    def merge_tropics(global_var, trop_var):
        aligned = trop_var.reindex(lat=ds_out.lat, method='nearest', tolerance=0.01)
        ds_out[global_var].loc[{'lat': trop_slice}] = aligned.sel(lat=trop_slice)

    merge_tropics('season_length', trop_len)
    merge_tropics('s1_start', trop_s1_start)
    merge_tropics('s1_end', trop_s1_end)
    merge_tropics('s2_start', trop_s2_start)
    merge_tropics('s2_end', trop_s2_end)

    encoding = {var: {'_FillValue': np.nan, 'dtype': 'float64', 'zlib': True} for var in ds_out.data_vars}
    ds_out.to_netcdf(output_file, encoding=encoding)
    print(f"Done! Unified results saved to {output_file}")

    stats = {'ds_merged': ds_out, 'trop_map': trop_map}
    return stats

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

def run_ensemble_analysis():
    all_members = find_ensemble_members(DATA_DIR)
    members = all_members
    
    print(f"Found {len(all_members)} total members. Testing on {len(members)}: {members}")
    
    if not members: return

    for member in members:
        print(f"\n========================================================")
        print(f" PROCESSING MEMBER: {member} (HISTORICAL RUN)")
        print(f"========================================================")
        
        hist_out_file = os.path.join(OUTPUT_DIR, f"ExtremeHeat_{member}_Hist_{ANALYSIS_HIST_START}_{ANALYSIS_HIST_END}.nc")
        
        if os.path.exists(hist_out_file):
            print(f"  [Skipping] Output file for {member} already exists.")
            continue
            
        hist_files = get_cesm_files(DATA_DIR, member, FILE_HIST_START, FILE_HIST_END, "BHISTsmbb")
        fut_files = get_cesm_files(DATA_DIR, member, FILE_FUT_START, FILE_FUT_END, "BSSP370smbb")
        
        if not hist_files or not fut_files:
            print(f"  [Skipping] Missing raw data files for member {member}")
            continue

        all_files = sorted(hist_files + fut_files)
        print(f"  Loading {len(all_files)} files...")
        
        try:
            ds = xr.open_mfdataset(all_files, combine='by_coords', parallel=True, chunks={'time': 365})
            
            buffer_hist = ANALYSIS_HIST_START - 1
            data_hist = ds[variable_name].sel(time=slice(str(buffer_hist), str(ANALYSIS_HIST_END)))
       
            thresh_p1 = calculate_threshold(data_hist, ANALYSIS_HIST_START, ANALYSIS_HIST_END, percentile=0.97)
            
            print("  Generating Bimodal Valley Maps for the historical timeline...")
            bimodal_results = map_tropical_bimodality(data_hist, thresh_p1) 
            modes_map_hist = bimodal_results['season_modes']
            valley_map_hist = bimodal_results['valley_doy']
            deep_month_map_hist = bimodal_results['deep_valley_month']
            
            print(f"\n  --- Running Historical Analysis for {member} ---")

            stats_hist = run_unified_global_analysis(
                data=data_hist,                           
                threshold=thresh_p1,                      
                start_year=ANALYSIS_HIST_START,           
                end_year=ANALYSIS_HIST_END,               
                output_file=hist_out_file,                
                ref_start_month_map=None,                 # Set to None to trigger mapping override
                bimodal_modes_map=modes_map_hist,         
                bimodal_valley_map=valley_map_hist,       
                bimodal_start_month_map=deep_month_map_hist, 
                extra_threshold=None                      
            )
            
            ds.close()
            
            print(f"\n  Successfully completed Historical Run for Member {member}.")
            
        except Exception as e:
            print(f"  [Critical Error] Member {member} failed: {e}")
            
    print("\n=== ALL MEMBERS PROCESSED ===")

# Execute the script
run_ensemble_analysis()